# CRNN: small-sample diagnostics and inference

This notebook loads the checkpoint from the latest controlled 16-word development experiment. It predicts romanization using a fresh-trained small CNN–BiLSTM–CTC model with ink cropping and group normalization. It does not use a 16-class word lookup: outputs are character sequences.

The training and validation images come from tyotakuki's synthetic dataset. Validation contains different images of the same 16 word types, and has been used to choose settings/checkpoints. These results do not measure recognition of unseen words or historical documents. The 250-image final test set remains untouched. See [planning_log.md](planning_log.md) for all failed and successful trials.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import torch
from PIL import Image
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents)
            if (p / "experiments/small_sample_model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.small_sample_model import load_model, predict, edit_distance

torch.set_num_threads(2)
CACHE = ROOT / "data/tyotakuki_pilot_16"
RUN = ROOT / "research_runs/tyotakuki_multiview_crop32_groupnorm"
CHECKPOINT = RUN / "best.pt"
model, checkpoint = load_model(CHECKPOINT)
manifest = json.loads((CACHE / "SOURCE_multiview.json").read_text())
results = json.loads((RUN / "results.json").read_text())
assert hashlib.sha256((CACHE / "SOURCE_multiview.json").read_bytes()).hexdigest() == results["source_manifest_sha256"]
print("Checkpoint:", CHECKPOINT)
print("Training updates at selected checkpoint:", checkpoint["step"])
print("Output alphabet:", "".join(checkpoint["char_to_id"]))

## Predict the cached training and validation images

Every cached file is checked against its recorded source checksum. Images are inverted to black on white when needed. A median-filtered mask estimates the foreground box, but the crop keeps original image pixels. The crop is resized to height 32, preserving aspect ratio, with minimum width 64. No horizontal mirror is applied.

In [ ]:
predictions = []
for row in manifest["records"]:
    path = CACHE / row["member"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["sha256"], path
    with Image.open(path) as image:
        word, box = predict(model, checkpoint, image)
    predictions.append({**row, "prediction": word, "box": box})

for split in ("train", "val"):
    selected = [r for r in predictions if r["split"] == split]
    exact = sum(r["prediction"] == r["romanization"] for r in selected)
    edits = sum(edit_distance(r["romanization"], r["prediction"]) for r in selected)
    characters = sum(len(r["romanization"]) for r in selected)
    print(f"{split}: {exact}/{len(selected)} words correct; CER={edits/characters:.2%}")

In [ ]:
for row in predictions:
    if row["split"] == "val":
        print(row["romanization"], "->", row["prediction"],
              "OK" if row["romanization"] == row["prediction"] else "ERROR")

In [ ]:
validation = [r for r in predictions if r["split"] == "val"]
fig, axes = plt.subplots(8, 2, figsize=(14, 18))
for axis, row in zip(axes.flat, validation):
    with Image.open(CACHE / row["member"]) as image:
        axis.imshow(image, cmap="gray")
    axis.set_title(f"Reference: {row['romanization']} | Prediction: {row['prediction']}")
    axis.axis("off")
plt.tight_layout()

## Reproduce the experiments

The completed histories and exact image manifests are under `research_runs/` and `data/tyotakuki_pilot_16/`. Scripts refuse to overwrite completed results. Choose new output names before rerunning a completed experiment.

From the workspace root:

```sh
research/manchu/bin/python research/experiments/ablate_preprocessing.py
research/manchu/bin/python research/experiments/train_multiview.py
research/manchu/bin/python research/experiments/train_multiview.py --groupnorm
```

The first script compares cropped 32px inputs, mirrored crops, and cropped 64px inputs. The last two compare models without and with group normalization on 60 training images of 16 word types. Original validation images remain excluded from training. These are development experiments selected after examining the data; a new seed and genuinely independent evaluation are required before making broad accuracy claims.